In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

dir = "/content/drive/MyDrive/HLT project/Project files/sentiment_annotated_with_texts.csv"


with open(dir, 'r') as f:
  sent = pd.read_csv(f, delimiter=',', quotechar='"')

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')
from itertools import chain
import nltk
import sklearn
import scipy.stats
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RandomizedSearchCV
import spacy

! pip install sklearn-crfsuite
import sklearn_crfsuite
from sklearn_crfsuite import scorers
from sklearn_crfsuite import metrics
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 13.3 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

Create the function used to tokenize and lemmatize our data

In [ ]:
nlp = spacy.load("en_core_web_sm")
def tokenize_and_lemmatize(text):
    doc = nlp(text)
    return [token.lemma_ for token in doc]

In [ ]:
all_text = sent[['title', 'true_sentiment']]
all_text['title'] = all_text['title'].apply(tokenize_and_lemmatize)
all_text_c = all_text.copy()
all_text_t = all_text.copy()

<ipython-input-5-b08e002fad9a>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_text['title'] = all_text['title'].apply(tokenize_and_lemmatize)


We apply the PoS tagging on the headlines

In [ ]:
all_text_t['title'] = all_text_c['title'].apply(lambda d: nltk.pos_tag(d))


In [ ]:
def split_sen(d):
  new_d = []
  sent = []
  for w in d:
    if w == '.':
      sent.append(w)
      sent = []
      new_d.append(sent)
    else:
      sent.append(w)
  return new_d

#all_text_c['text'] = all_text['text'].apply(lambda d: split_sen(d))


In [ ]:
all_text_t[:4]

,title,true_sentiment
0,"[(euro, NN), (to, TO), (benefit, VB), (from, I...",Positive
1,"[(EURCHF, NNP), (Trend, NNP), (higher, JJR), (...",Positive
2,"[(do, VB), (a, DT), (jump, NN), (in, IN), (EUR...",Neutral
3,"[(EURCHF, NNP), (could, MD), (extend, VB), (it...",Positive


We load the NRC-VAD Lexicon

In [ ]:
txt_sent = "/content/drive/MyDrive/HLT project/Project files/NRC-VAD-Lexicon.txt"

data = {}

with open(txt_sent, "r") as f:
  t = f.read()
  for si in t.split('\n'):
    si_w = (si +'\t').split('\t')
    if si_w[0] != '':
      data[si_w[0]] = list(map(lambda fl: float(fl), si_w[1:-1]))
    else:
      pass

Here we load the SentiWordNet to retrieve the positive and negative scores and use them as a feature

In [ ]:
import re

txt_sent = "/content/drive/MyDrive/HLT project/Project files/SentiWordNet_3.0.0.txt"
senti_word = {}
regex = '0(\.\d+)?\s0(\.\d+)?\s([a-z_\-]+#\d+\s)+'
pattern = re.compile(regex)


with open(txt_sent, "r") as f:
  t = f.read()
  for si in t.split("\n"):
    if len(si) > 0 and si[0] != '#':
      matched = pattern.search(si)
      if matched:
        val = matched.group(0).split('\t')
        scor = [float(val[0]), float(val[1])]
        wors = re.split('#\d+\s?', val[2])
        wors.pop(-1)
        for wr in wors:
          senti_word[wr] = scor

df_senti = pd.DataFrame(senti_word, index=["pos_score", "negative_score"])

In [ ]:

for w,v in data.items():
  if len(v) < 3:
    print(w)
df2 = pd.DataFrame(data, index=['valence', 'arousal', 'dominance'])



This is the function used to retrieve the features from the tokens of every headline present in our dataset.

In [ ]:
import string

def word2features(sent, i):
    word = sent[i][0]
    postag = sent[i][1]

    regex= "^\+|-\d+\.\d+"
    pattern = re.compile(regex)

    add_score = False




    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'len(word)': len(word),
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        'postag': postag,
        'postag[:2]': postag[:2],
        'word.ispunctuation': (word in string.punctuation),
    }
    try:
      sc = df_senti[word.lower()]
      features.update({
              'word.posScore': sc[0],
              'word.negScore': sc[1],
          })
      add_score = True
    except KeyError:
      pass

    if pattern.match(word) is not None:
      features['interests'] = word
    else:
      features['interests'] = 0

    if i > 0: #previous
        word1 = sent[i-1][0]
        postag1 = sent[i-1][1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
            '-1:postag': postag1,
            '-1:postag[:2]': postag1[:2],
        })
        com_wrod = word1.lower()+ '_'+word.lower()
        try:
          sc = df_senti[com_wrod]
          add_score = True
          features.update({
              '-1:word.posScore': sc[0],
              '-1:word.negScore': sc[1],
          })
        except KeyError:
          pass
    else:
        features['BOS'] = True

    if i < len(sent)-1:#next
        word1 = sent[i+1][0]
        postag1 = sent[i+1][1]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
            '+1:postag': postag1,
            '+1:postag[:2]': postag1[:2],
        })

        com_wrod = word1.lower()+ '_'+word.lower()
        try:
          sc = df_senti[com_wrod]
          add_score = True
          features.update({
              '+1:word.posScore': sc[0],
              '+1:word.negScore': sc[1],
          })
        except KeyError:
          pass
    else:
        features['EOS'] = True

    try:

      features['valence'] = df2.iloc[0][word]
      features['arousal'] = df2.iloc[1][word]
      features['dominance'] = df2.iloc[2][word]
    except KeyError:
      pass

    return features


def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]


def sent2labels(sent):
    return [label for token, postag, label in sent]

def sent2tokens(sent):
    return [token for token, postag, label in sent]

In [ ]:
all_X = [sent2features(s) for s in all_text_t['title']]
print(all_X[:3])

[[{'bias': 1.0, 'word.lower()': 'euro', 'len(word)': 4, 'word[-3:]': 'uro', 'word[-2:]': 'ro', 'word.isupper()': False, 'word.istitle()': False, 'word.isdigit()': False, 'postag': 'NN', 'postag[:2]': 'NN', 'word.ispunctuation': False, 'word.posScore': 0.0, 'word.negScore': 0.0, 'interests': 0, 'BOS': True, '+1:word.lower()': 'to', '+1:word.istitle()': False, '+1:word.isupper()': False, '+1:postag': 'TO', '+1:postag[:2]': 'TO', 'valence': 0.68, 'arousal': 0.441, 'dominance': 0.521}, {'bias': 1.0, 'word.lower()': 'to', 'len(word)': 2, 'word[-3:]': 'to', 'word[-2:]': 'to', 'word.isupper()': False, 'word.istitle()': False, 'word.isdigit()': False, 'postag': 'TO', 'postag[:2]': 'TO', 'word.ispunctuation': False, 'interests': 0, '-1:word.lower()': 'euro', '-1:word.istitle()': False, '-1:word.isupper()': False, '-1:postag': 'NN', '-1:postag[:2]': 'NN', '+1:word.lower()': 'benefit', '+1:word.istitle()': False, '+1:word.isupper()': False, '+1:postag': 'VB', '+1:postag[:2]': 'VB'}, {'bias': 1.0,

In [ ]:
import numpy as np

all_Y = [list(np.repeat(l, len(all_X[i]))) for i,l in enumerate(all_text_t['true_sentiment'])]

We split the datas as follows: 80% for the training set and 20% for the test set

In [ ]:
all_l = len(all_X)

train_ind = int(all_l * 0.8)
X_train = all_X[0:train_ind]
X_test = all_X[train_ind:all_l]

y_train = all_Y[0:train_ind]
y_test = all_Y[train_ind:all_l]
# print(X_test[0])
print(len(X_test))
# print(y_test[0])

459


We create and train our Conditional Random Field model

In [ ]:
%%time
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',# Limited-memory BFGS is an optimization algorithm for parameter estimation
    c1=0.2,
    c2=0.2,
    max_iterations=300,
    all_possible_transitions=True,
    epsilon=1e-4,
    delta=1e-5,
    verbose=True
)
try:
    crf.fit(X_train, y_train)
except AttributeError:
    pass

loading training data to CRFsuite: 100%|██████████| 1832/1832 [00:00<00:00, 4711.83it/s]



Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 16316
Seconds required: 0.111

L-BFGS optimization
c1: 0.200000
c2: 0.200000
num_memories: 6
max_iterations: 300
epsilon: 0.000100
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

Iter 1   time=0.04  loss=22179.21 active=15054 feature_norm=0.50
Iter 2   time=0.02  loss=18732.62 active=15895 feature_norm=0.79
Iter 3   time=0.02  loss=7025.71  active=15739 feature_norm=2.96
Iter 4   time=0.04  loss=6471.14  active=16216 feature_norm=3.29
Iter 5   time=0.03  loss=5558.76  active=16254 feature_norm=3.65
Iter 6   time=0.03  loss=4690.54  active=16225 feature_norm=3.98
Iter 7   time=0.02  loss=4544.80  active=16229 feature_norm=4.50
Iter 8   time=0.02  loss=3743.32  active=16134 feature_norm=4.94
Iter 9   time=0.03  loss=3163.34  active=16183 feature_norm=5.11
Iter 10  time=

In [ ]:
labels = list(crf.classes_)
labels


['Positive', 'Neutral', 'Negative']

We make Predictions on test set

In [ ]:
y_pred = crf.predict(X_test)


In [ ]:
metrics.flat_f1_score(y_test, y_pred,
                      average='weighted', labels=labels)

0.6986949530829621

Here we print the evaluation metrics using the skmetrics.classification_report

In [ ]:
from sklearn_crfsuite.utils import flatten
from sklearn import metrics as skmetrics

sorted_labels = sorted(
    labels,
    key=lambda name: (name[1:], name[0])
)
y_test_flat = flatten(y_test)
y_pred_flat = flatten(y_pred)
print(skmetrics.classification_report(y_test_flat, y_pred_flat, labels=sorted_labels, digits=2))


              precision    recall  f1-score   support

    Negative       0.69      0.64      0.66      1559
     Neutral       0.69      0.74      0.71      1874
    Positive       0.72      0.71      0.72      1711

    accuracy                           0.70      5144
   macro avg       0.70      0.70      0.70      5144
weighted avg       0.70      0.70      0.70      5144



In [ ]:
print(crf.size_)

455008


In [ ]:
!pip freeze > '/content/drive/MyDrive/HLT project/requirements_CRF.txt'